            # LA-M2MRF Reproduction Workflow

            這份 notebook 旨在把 LA-M2MRF 在 IDRiD 上的訓練、評估、對照與視覺化流程整理成一份可重跑、可檢查、可延伸的重現工作流。

            ---

            ## Workflow Map

            | 段落 | 主題 | 核心內容 |
            | --- | --- | --- |
            | 1 | 環境需求與安裝說明 | 確認依賴、資料路徑、官方 checkpoint |
            | 2 | 原始 M2MRF-C baseline | `bs=3, fp16` scratch 訓練 / 驗證 |
            | 3 | 官方 checkpoint finetune 對照 | Baseline、A、B、A+B |
            | 4 | 全量 scratch 對照 | Baseline、A、B、A+B |

            ## Design Principles

            - Reuse existing checkpoints whenever possible.
            - Train only when checkpoints are missing and `TRAIN_IF_MISSING=True`.
            - Keep generated figures under `work_dirs/la_m2mrf/notebook_artifacts/`.
            - Keep model weights and logs inside their original `work_dirs/la_m2mrf/...` experiment folders.

            > Recommended flow:
            > run Section 1 first, confirm environment and assets, then decide whether to reproduce from existing checkpoints or launch missing runs.
            


In [ ]:
            from pathlib import Path
            import os
            import sys

            def find_repo_root(start: Path) -> Path:
                for candidate in [start, *start.parents]:
                    if (candidate / 'tools' / 'la_m2mrf_notebook_utils.py').is_file():
                        return candidate
                raise FileNotFoundError('Cannot locate repo root from current working directory.')

            REPO_ROOT = find_repo_root(Path.cwd()).resolve()
            os.chdir(REPO_ROOT)
            sys.path.insert(0, str(REPO_ROOT / 'tools'))

            import pandas as pd
            from IPython.display import Image, Markdown, display
            import la_m2mrf_notebook_utils as nb

            ARTIFACT_ROOT = nb.ensure_artifact_root()
            FONT_PATH = nb.configure_plot_fonts()

            def _env_bool(name: str, default: bool) -> bool:
                raw = os.environ.get(name)
                if raw is None:
                    return default
                return raw.strip().lower() not in {'0', 'false', 'no', 'off'}

            RUN_INSTALL = _env_bool('LA_M2MRF_RUN_INSTALL', False)
            TRAIN_IF_MISSING = _env_bool('LA_M2MRF_TRAIN_IF_MISSING', True)
            FORCE_RETRAIN_SECTION2 = _env_bool('LA_M2MRF_FORCE_RETRAIN_SECTION2', False)
            FORCE_RETRAIN_SECTION3 = _env_bool('LA_M2MRF_FORCE_RETRAIN_SECTION3', False)
            FORCE_RETRAIN_SECTION4 = _env_bool('LA_M2MRF_FORCE_RETRAIN_SECTION4', False)
            FORCE_REEVAL = _env_bool('LA_M2MRF_FORCE_REEVAL', False)
            SAMPLE_IMAGE = None

            print('Repo root:', REPO_ROOT)
            print('Artifact root:', ARTIFACT_ROOT)
            print('Chinese font:', FONT_PATH if FONT_PATH else 'fallback / English-only rendering')
            


            ## 1. Environment, Dependencies, and Assets

            在進入任何訓練或推論之前，先確認這份 workflow 所依賴的執行環境、資料位置與官方 checkpoint 都能被正確辨識。

            這裡有兩層相依性需要先分清楚：

            - 第一層是研究程式本身的相容性目標，例如 `MMSegmentation 0.8.0` 與舊版 `MMCV`。
            - 第二層是目前這台機器上已經驗證可用的實際 Python / CUDA / PyTorch 基底環境。

            本節會完成三件事：

            - 顯示 notebook 建議的環境與套件需求
            - 顯示可直接複製執行的安裝命令
            - 檢查 IDRiD 資料集位置與官方 checkpoint 是否存在

            > 如果只是要重跑既有結果，通常不需要重新安裝。
            > 如果要在乾淨環境完整重現，再把 `RUN_INSTALL` 打開。
            


In [ ]:
            print(nb.environment_dependency_markdown())
            print('\nInstall commands:\n')
            print(nb.format_commands(nb.install_commands()))

            print('\nDataset root:', nb.dataset_root())
            print('Dataset exists:', nb.dataset_exists())

            try:
                print('Official checkpoint:', nb.find_official_checkpoint())
            except FileNotFoundError as exc:
                print(exc)

            if RUN_INSTALL:
                import subprocess
                install_script = nb.format_commands(nb.install_commands())
                print('\nRunning install script:\n')
                print(install_script)
                completed = subprocess.run(
                    ['bash', '-lc', install_script],
                    cwd=str(REPO_ROOT),
                    text=True,
                    check=False)
                if completed.returncode != 0:
                    raise RuntimeError('Install script failed.')
            


            ## 2. Baseline Scratch Reproduction (`bs=3`, `fp16`)

            本節對應「不加入 LA-M2MRF 改動，只保留原始 M2MRF-C baseline 方法」的單卡 `bs=3, fp16` scratch 流程。它是後續所有改動的參照基線。

            Workflow:

            - 先找到已存在的 baseline scratch checkpoint；如果已存在，就直接沿用
            - 如果找不到，而且 `TRAIN_IF_MISSING=True`，就用 `configs/la_m2mrf/fcn_hr48-M2MRF-C_40k_idrid_official_scratch_bs3_fp16.py` 啟動訓練
            - 整理 baseline 訓練 loss 圖
            - 評估並對照官方釋出的 checkpoint 與本地 `bs=3, fp16` baseline scratch 結果
            - 產生 qualitative inference 圖，讓老師直接看到實際分割輸出

            Outputs:

            - baseline checkpoint 重用或訓練結果
            - 官方 checkpoint 與本地 baseline 的指標對照表
            - baseline loss curve
            - baseline qualitative inference 圖

            > 這一節的目標是建立可重複操作、可檢查、可推論的 baseline scratch 流程，不是直接把任何本地結果宣稱為官方重現完成。
            


In [ ]:
            sample_image = Path(SAMPLE_IMAGE).expanduser().resolve() if SAMPLE_IMAGE else nb.default_sample_paths()[0]
            official_ckpt = nb.find_official_checkpoint()

            baseline_ckpt = nb.ensure_training(
                nb.BASELINE_SCRATCH_SPEC,
                train_if_missing=TRAIN_IF_MISSING,
                force_train=FORCE_RETRAIN_SECTION2)
            nb.ensure_loss_plots(nb.BASELINE_SCRATCH_SPEC.workdir)

            official_metrics = nb.evaluate_checkpoint(
                label='Official released checkpoint',
                config_path=nb.repo_path('configs', 'la_m2mrf', 'fcn_hr48-M2MRF-C_40k_idrid_baseline_copy.py'),
                checkpoint_path=official_ckpt,
                cache_key='official_released_checkpoint',
                force_eval=FORCE_REEVAL)
            baseline_metrics = nb.collect_metrics(
                nb.BASELINE_SCRATCH_SPEC,
                checkpoint_path=baseline_ckpt,
                force_eval=FORCE_REEVAL)

            baseline_rows = nb.summarize_rows([official_metrics, baseline_metrics])
            display(pd.DataFrame(baseline_rows))

            baseline_panel = ARTIFACT_ROOT / 'section2_baseline_prediction_panel.png'
            nb.build_prediction_panel(
                [
                    ('Official checkpoint / 官方權重', nb.repo_path('configs', 'la_m2mrf', 'fcn_hr48-M2MRF-C_40k_idrid_baseline_copy.py'), official_ckpt),
                    ('Local bs3 fp16 baseline / 本地基線', nb.BASELINE_SCRATCH_SPEC.config, baseline_ckpt),
                ],
                baseline_panel,
                sample_image=sample_image)

            display(Markdown('### Baseline loss curve'))
            display(Image(filename=str(nb.BASELINE_SCRATCH_SPEC.workdir / 'plots' / 'loss_iter.png')))

            display(Markdown('### Baseline qualitative inference'))
            display(Image(filename=str(baseline_panel)))
            


            ## 3. Finetune Ablations from the Official Checkpoint

            這一節聚焦於從原始論文提供的官方 checkpoint 出發，進行較短的 finetune，觀察兩個方法改動是否帶來可見差異。

            Variants:

            - Baseline：只做官方 checkpoint continuation，不加新方法
            - A：`Sampling`，也就是 Lesion-Aware Patch Sampling / 病灶感知區塊採樣
            - B：`Weighted Dice`，也就是 Weighted Binary Dice Loss / 加權二元 Dice 損失
            - A+B：同時啟用 A 與 B

            This section produces:

            - 方法示意圖：原圖、A、B、A+B 的處理概念
            - quantitative 表格：Baseline、A、B、A+B 的指標對照
            - loss 視覺化：四組 finetune 的訓練曲線
            - inference qualitative：Original、Baseline、A、B、A+B 的模型輸出對照

            > If a finetune checkpoint already exists, the workflow reuses it directly and only refreshes tables, curves, and qualitative outputs.
            


In [ ]:
            finetune_method_preview = ARTIFACT_ROOT / 'section3_method_preview.png'
            nb.build_method_preview(finetune_method_preview)
            display(Image(filename=str(finetune_method_preview)))
            


In [ ]:
            finetune_metrics = []
            finetune_items = []
            finetune_missing = []

            for spec in nb.FINETUNE_EXPERIMENTS:
                try:
                    checkpoint = nb.ensure_training(
                        spec,
                        train_if_missing=TRAIN_IF_MISSING,
                        force_train=FORCE_RETRAIN_SECTION3)
                    nb.ensure_loss_plots(spec.workdir)
                    finetune_metrics.append(
                        nb.collect_metrics(spec, checkpoint_path=checkpoint, force_eval=FORCE_REEVAL))
                    finetune_items.append((spec.label, spec.config, checkpoint))
                except FileNotFoundError as exc:
                    finetune_missing.append((spec.label, str(exc)))

            if finetune_metrics:
                display(pd.DataFrame(nb.summarize_rows(finetune_metrics)))
            if finetune_missing:
                display(Markdown('### Missing finetune checkpoints'))
                display(pd.DataFrame(finetune_missing, columns=['Experiment', 'Reason']))

            finetune_loss_grid = ARTIFACT_ROOT / 'section3_finetune_loss_grid.png'
            finetune_pred_panel = ARTIFACT_ROOT / 'section3_finetune_prediction_panel.png'

            if finetune_metrics:
                nb.build_loss_grid(
                    [spec for spec in nb.FINETUNE_EXPERIMENTS if any(row['label'] == spec.label for row in finetune_metrics)],
                    finetune_loss_grid,
                    title='Finetune experiments / 微調實驗：loss overview')
                nb.build_prediction_panel(
                    finetune_items,
                    finetune_pred_panel,
                    sample_image=sample_image)

                display(Markdown('### Finetune loss overview'))
                display(Image(filename=str(finetune_loss_grid)))

                display(Markdown('### Finetune qualitative inference'))
                display(Image(filename=str(finetune_pred_panel)))
            


            ## 4. Full Scratch Ablations (`bs=3`, `fp16`)

            這一節將 Baseline、A、B、A+B 全部放到 `bs=3, fp16` scratch 設定下重新訓練，用來比較「不依賴官方 checkpoint」時各種方法改動的效果。

            Questions addressed here:

            - 如果不依賴官方 checkpoint，而是從頭開始訓練，A 與 B 各自有沒有作用？
            - A 與 B 同時啟用時，是否和單獨使用時呈現不同趨勢？
            - 與 scratch baseline 相比，哪一個組合在目前這個 workspace 的設定下表現更穩定？

            Outputs:

            - 逐組 checkpoint 檢查與必要時的訓練
            - Baseline / A / B / A+B 的 metrics 表格
            - 每組的 loss 圖與 qualitative inference 圖

            > `40k` scratch runs are substantially more expensive than finetune runs. If checkpoints are missing and `TRAIN_IF_MISSING=True`, expect this section to take much longer.
            


In [ ]:
            scratch_method_preview = ARTIFACT_ROOT / 'section4_method_preview.png'
            nb.build_method_preview(scratch_method_preview)
            display(Image(filename=str(scratch_method_preview)))
            


In [ ]:
            scratch_metrics = []
            scratch_items = []
            scratch_missing = []

            for spec in nb.SCRATCH_EXPERIMENTS:
                try:
                    checkpoint = nb.ensure_training(
                        spec,
                        train_if_missing=TRAIN_IF_MISSING,
                        force_train=FORCE_RETRAIN_SECTION4)
                    nb.ensure_loss_plots(spec.workdir)
                    scratch_metrics.append(
                        nb.collect_metrics(spec, checkpoint_path=checkpoint, force_eval=FORCE_REEVAL))
                    scratch_items.append((spec.label, spec.config, checkpoint))
                except FileNotFoundError as exc:
                    scratch_missing.append((spec.label, str(exc)))

            if scratch_metrics:
                display(pd.DataFrame(nb.summarize_rows(scratch_metrics)))
            if scratch_missing:
                display(Markdown('### Missing scratch checkpoints'))
                display(pd.DataFrame(scratch_missing, columns=['Experiment', 'Reason']))

            scratch_loss_grid = ARTIFACT_ROOT / 'section4_scratch_loss_grid.png'
            scratch_pred_panel = ARTIFACT_ROOT / 'section4_scratch_prediction_panel.png'

            if scratch_metrics:
                nb.build_loss_grid(
                    [spec for spec in nb.SCRATCH_EXPERIMENTS if any(row['label'] == spec.label for row in scratch_metrics)],
                    scratch_loss_grid,
                    title='Scratch experiments / 從零訓練實驗：loss overview')
                nb.build_prediction_panel(
                    scratch_items,
                    scratch_pred_panel,
                    sample_image=sample_image)

                display(Markdown('### Scratch loss overview'))
                display(Image(filename=str(scratch_loss_grid)))

                display(Markdown('### Scratch qualitative inference'))
                display(Image(filename=str(scratch_pred_panel)))
            


            ## Closing Notes

            完成整份 workflow 後，通常會得到三類可直接檢查的產物：

            - 訓練產物：各實驗的 checkpoint、log、loss 曲線
            - 定量結果：Baseline / A / B / A+B 的指標表格
            - 定性結果：方法示意圖與 inference qualitative 圖

            Practical usage:

            - 如果只想快速檢查結果，保留既有 checkpoint，讓 notebook 直接重用現成實驗。
            - 如果要完整重現流程，先確認階段一，再允許 notebook 補跑缺少的訓練。
            - 如果只想看某一段，可以單獨執行該段落，不一定要每次都從頭完整跑完。

            > 這份 notebook 的定位不是簡單展示，而是一份面向重現、分析與延伸實驗的工作流工具。
            
